# IBM Bob Hackathon: BB84 Quantum Key Distribution Educational Notebook

This notebook is a **single-file, ready-to-run Qiskit prototype** for a BB84 Quantum Key Distribution educational interface. It is designed for the IBM Bob Hackathon workflow: it teaches the protocol, runs a clean BB84 simulation, introduces Eve with intercept-resend, visualizes QBER, includes an optional noisy simulator, and provides a safe path for small IBM Quantum hardware demonstrations.

> **Hackathon framing:** Alice and Bob use randomly selected quantum bases to establish a shared secret key. Eve cannot copy unknown quantum states perfectly, and measuring in the wrong basis disturbs the transmission. The educational goal is to make this disturbance visible through **Quantum Bit Error Rate**, or **QBER**.

| Section | Purpose | Hackathon Value |
|---|---|---|
| Setup | Installs Qiskit, Aer, NumPy, Matplotlib, and optional IBM Runtime | Makes the notebook runnable in Google Colab or locally |
| BB84 Core | Implements Alice encoding, Bob measurement, sifting, QBER, and privacy-safe final key extraction | Gives the project a real quantum-computing foundation |
| Eve Demo | Adds intercept-resend attack and shows QBER rising toward the expected 25 percent range | Creates the central educational reveal |
| Noise Demo | Adds a simple hardware-inspired noise option | Helps explain real device limitations |
| IBM Quantum Path | Provides optional Qiskit Runtime code with secure token handling | Aligns the project with IBM Quantum ecosystem |

In [ ]:
# If running in Google Colab, uncomment this installation cell.
# In a local environment, run the same command in your terminal.

# !pip install -q qiskit qiskit-aer qiskit-ibm-runtime numpy matplotlib pandas

In [ ]:
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

RNG_SEED = 84
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

Z_BASIS = "Z"
X_BASIS = "X"
BASES = [Z_BASIS, X_BASIS]
SECURITY_THRESHOLD = 0.11  # common teaching threshold; production QKD needs full security analysis
SIMULATOR = AerSimulator(seed_simulator=RNG_SEED)

print("Notebook ready. Qiskit Aer simulator initialized.")

## 1. BB84 Concept Map

BB84 uses two conjugate bases. In this notebook, **Z** means the computational basis and **X** means the Hadamard basis.

| Bit | Basis | Quantum State | Circuit Preparation |
|---:|---|---|---|
| 0 | Z | `|0>` | no gate |
| 1 | Z | `|1>` | `X` |
| 0 | X | `|+>` | `H` |
| 1 | X | `|->` | `X`, then `H` |

Alice sends qubits. Bob measures each qubit in a random basis. Alice and Bob publicly compare only their bases, not their bit values. Matching-basis positions become the **sifted key**. A random sample of the sifted key is revealed to estimate QBER. If QBER is too high, the key is rejected.

In [ ]:
@dataclass
class BB84Result:
    n_qubits: int
    eve_strategy: str
    channel_noise: float
    alice_bits: List[int]
    alice_bases: List[str]
    bob_bases: List[str]
    bob_bits: List[int]
    sifted_indices: List[int]
    sample_positions: List[int]
    alice_sample: List[int]
    bob_sample: List[int]
    alice_final_key: List[int]
    bob_final_key: List[int]
    qber: float
    accepted: bool

    @property
    def sifted_key_length(self) -> int:
        return len(self.sifted_indices)

    @property
    def final_key_length(self) -> int:
        return len(self.alice_final_key)

    @property
    def keys_match(self) -> bool:
        return self.alice_final_key == self.bob_final_key

    def summary(self) -> Dict[str, object]:
        return {
            "n_qubits": self.n_qubits,
            "eve_strategy": self.eve_strategy,
            "channel_noise": self.channel_noise,
            "sifted_key_length": self.sifted_key_length,
            "sample_size": len(self.sample_positions),
            "final_key_length": self.final_key_length,
            "qber_percent": round(100 * self.qber, 2),
            "accepted": self.accepted,
            "final_keys_match": self.keys_match,
        }


def random_bits(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]


def random_bases(n: int) -> List[str]:
    return [random.choice(BASES) for _ in range(n)]

## 2. Qiskit Circuit Primitives

The next functions are deliberately small and readable so they can be reused in a Streamlit educational interface. Each transmitted bit is represented by a one-qubit circuit. This is slower than a fully vectorized NumPy model but much clearer for teaching and circuit drawing.

In [ ]:
def encode_bit(bit: int, basis: str) -> QuantumCircuit:
    """Encode one classical bit into one BB84 qubit.

    Args:
        bit: Classical bit, either 0 or 1.
        basis: 'Z' for computational basis or 'X' for Hadamard basis.

    Returns:
        A one-qubit Qiskit circuit without measurement.
    """
    if bit not in (0, 1):
        raise ValueError("bit must be 0 or 1")
    if basis not in BASES:
        raise ValueError("basis must be 'Z' or 'X'")

    qc = QuantumCircuit(1, 1, name=f"encode_{bit}_{basis}")
    if bit == 1:
        qc.x(0)
    if basis == X_BASIS:
        qc.h(0)
    return qc


def add_measurement_in_basis(circuit: QuantumCircuit, basis: str) -> QuantumCircuit:
    """Return a measured copy of a one-qubit circuit in Bob or Eve's chosen basis."""
    if basis not in BASES:
        raise ValueError("basis must be 'Z' or 'X'")
    qc = circuit.copy()
    if basis == X_BASIS:
        qc.h(0)
    qc.measure(0, 0)
    return qc


def measure_one_shot(circuit: QuantumCircuit, basis: str, simulator=AerSimulator(seed_simulator=RNG_SEED)) -> int:
    """Measure a one-qubit circuit once and return a classical bit."""
    measured = add_measurement_in_basis(circuit, basis)
    tqc = transpile(measured, simulator, seed_transpiler=RNG_SEED)
    result = simulator.run(tqc, shots=1).result()
    counts = result.get_counts()
    return int(max(counts, key=counts.get))


def draw_encoding_examples():
    """Draw all four BB84 encoding circuits for classroom explanation."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 5))
    examples = [(0, Z_BASIS), (1, Z_BASIS), (0, X_BASIS), (1, X_BASIS)]
    for ax, (bit, basis) in zip(axes.ravel(), examples):
        qc = encode_bit(bit, basis)
        ax.axis("off")
        ax.set_title(f"bit={bit}, basis={basis}")
        ax.text(0.0, 0.5, qc.draw(output="text"), family="monospace", fontsize=10)
    plt.tight_layout()
    return fig

fig = draw_encoding_examples()
plt.show()

## 3. Eve Strategies and Channel Noise

This notebook implements three channel modes.

| Mode | Meaning | Educational Outcome |
|---|---|---|
| `none` | No eavesdropper | QBER should be near 0 on an ideal simulator |
| `intercept_resend` | Eve measures in a random basis and resends a new qubit | QBER tends toward about 25 percent on sifted bits |
| channel noise | Randomly flips a transmitted qubit with configurable probability | Demonstrates that real hardware and noisy channels can also raise QBER |

In [ ]:
def apply_simple_bit_flip_noise(circuit: QuantumCircuit, probability: float) -> QuantumCircuit:
    """Apply an educational bit-flip noise event before Bob receives the qubit.

    This is intentionally simple: with probability p, an X gate is added. It is not a full
    physical noise model, but it is easy to explain in a hackathon demo.
    """
    if not 0.0 <= probability <= 1.0:
        raise ValueError("probability must be between 0 and 1")
    noisy = circuit.copy()
    if random.random() < probability:
        noisy.x(0)
    return noisy


def intercept_resend(circuit: QuantumCircuit) -> Tuple[QuantumCircuit, str, int]:
    """Simulate Eve's intercept-resend attack.

    Eve chooses a random basis, measures the incoming qubit, then prepares a new qubit
    matching her measurement result and basis. If Eve guessed the wrong basis, she has
    disturbed the state sent to Bob.
    """
    eve_basis = random.choice(BASES)
    eve_bit = measure_one_shot(circuit, eve_basis)
    resent = encode_bit(eve_bit, eve_basis)
    return resent, eve_basis, eve_bit

## 4. Full BB84 Protocol Simulation

The function below performs the complete protocol: Alice preparation, optional Eve attack, optional noise, Bob measurement, basis sifting, QBER sampling, and final key extraction.

In [ ]:
def run_bb84(
    n_qubits: int = 256,
    sample_ratio: float = 0.25,
    eve_strategy: str = "none",
    channel_noise: float = 0.0,
    threshold: float = SECURITY_THRESHOLD,
) -> BB84Result:
    """Run a complete BB84 simulation with optional Eve and channel noise."""
    if n_qubits <= 0:
        raise ValueError("n_qubits must be positive")
    if not 0.0 <= sample_ratio <= 1.0:
        raise ValueError("sample_ratio must be between 0 and 1")
    if eve_strategy not in ("none", "intercept_resend"):
        raise ValueError("eve_strategy must be 'none' or 'intercept_resend'")

    alice_bits = random_bits(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)
    bob_bits: List[int] = []

    for i in range(n_qubits):
        qubit = encode_bit(alice_bits[i], alice_bases[i])

        if eve_strategy == "intercept_resend":
            qubit, _, _ = intercept_resend(qubit)

        qubit = apply_simple_bit_flip_noise(qubit, channel_noise)
        bob_bits.append(measure_one_shot(qubit, bob_bases[i]))

    sifted_indices = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_sifted = [alice_bits[i] for i in sifted_indices]
    bob_sifted = [bob_bits[i] for i in sifted_indices]

    sample_size = max(1, int(len(sifted_indices) * sample_ratio)) if sifted_indices else 0
    sample_positions = sorted(random.sample(range(len(sifted_indices)), k=min(sample_size, len(sifted_indices)))) if sample_size else []

    alice_sample = [alice_sifted[pos] for pos in sample_positions]
    bob_sample = [bob_sifted[pos] for pos in sample_positions]
    mismatches = sum(a != b for a, b in zip(alice_sample, bob_sample))
    qber = mismatches / len(sample_positions) if sample_positions else 0.0

    final_positions = [pos for pos in range(len(sifted_indices)) if pos not in set(sample_positions)]
    alice_final = [alice_sifted[pos] for pos in final_positions]
    bob_final = [bob_sifted[pos] for pos in final_positions]

    return BB84Result(
        n_qubits=n_qubits,
        eve_strategy=eve_strategy,
        channel_noise=channel_noise,
        alice_bits=alice_bits,
        alice_bases=alice_bases,
        bob_bases=bob_bases,
        bob_bits=bob_bits,
        sifted_indices=sifted_indices,
        sample_positions=sample_positions,
        alice_sample=alice_sample,
        bob_sample=bob_sample,
        alice_final_key=alice_final,
        bob_final_key=bob_final,
        qber=qber,
        accepted=qber <= threshold,
    )

clean_run = run_bb84(n_qubits=256, eve_strategy="none")
eve_run = run_bb84(n_qubits=256, eve_strategy="intercept_resend")

pd.DataFrame([clean_run.summary(), eve_run.summary()])

## 5. Explainable Results Table

In an ideal simulator with no Eve and no channel noise, Alice and Bob should agree whenever their bases match. With intercept-resend, Eve's wrong-basis measurements introduce errors. Because only about half of Alice and Bob's bases match, and Eve chooses the wrong basis about half the time, the expected QBER on sifted bits is approximately **25 percent**.

In [ ]:
def show_protocol_trace(result: BB84Result, rows: int = 24) -> pd.DataFrame:
    """Return the first rows of the protocol as an explainable table."""
    data = []
    sifted_set = set(result.sifted_indices)
    for i in range(min(rows, result.n_qubits)):
        data.append({
            "index": i,
            "Alice bit": result.alice_bits[i],
            "Alice basis": result.alice_bases[i],
            "Bob basis": result.bob_bases[i],
            "Bob bit": result.bob_bits[i],
            "basis match?": i in sifted_set,
            "sampled for QBER?": (result.sifted_indices.index(i) in result.sample_positions) if i in sifted_set else False,
        })
    return pd.DataFrame(data)

print("Clean run summary:", clean_run.summary())
display(show_protocol_trace(clean_run))

print("Eve run summary:", eve_run.summary())
display(show_protocol_trace(eve_run))

## 6. Monte Carlo Sweep: QBER with and without Eve

A single run can fluctuate because the protocol is random. A sweep makes the pattern clear for judges and learners.

In [ ]:
def monte_carlo_sweep(trials: int = 40, n_qubits: int = 256, noise_values: Optional[List[float]] = None) -> pd.DataFrame:
    if noise_values is None:
        noise_values = [0.0, 0.01, 0.03, 0.05, 0.08]
    records = []
    for noise in noise_values:
        for eve in ["none", "intercept_resend"]:
            qbers = []
            final_lengths = []
            accepted = []
            for _ in range(trials):
                res = run_bb84(n_qubits=n_qubits, eve_strategy=eve, channel_noise=noise)
                qbers.append(res.qber)
                final_lengths.append(res.final_key_length)
                accepted.append(res.accepted)
            records.append({
                "noise": noise,
                "eve_strategy": eve,
                "mean_qber_percent": 100 * float(np.mean(qbers)),
                "std_qber_percent": 100 * float(np.std(qbers)),
                "mean_final_key_length": float(np.mean(final_lengths)),
                "acceptance_rate": float(np.mean(accepted)),
            })
    return pd.DataFrame(records)

sweep_df = monte_carlo_sweep(trials=30, n_qubits=256)
display(sweep_df)

fig, ax = plt.subplots(figsize=(9, 5))
for eve, group in sweep_df.groupby("eve_strategy"):
    ax.errorbar(group["noise"], group["mean_qber_percent"], yerr=group["std_qber_percent"], marker="o", capsize=4, label=eve)
ax.axhline(SECURITY_THRESHOLD * 100, color="red", linestyle="--", label="security threshold")
ax.set_title("BB84 QBER: clean channel vs intercept-resend Eve")
ax.set_xlabel("Simple bit-flip noise probability")
ax.set_ylabel("Mean QBER (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Optional: A Faster Density-Matrix Teaching Simulator

For a web frontend, thousands of one-qubit Qiskit circuits can be slow. The next cell provides a compact density-matrix simulator. It preserves the educational logic while running much faster. This is a useful foundation if you later connect the notebook logic to a Streamlit app.

In [ ]:
KET0 = np.array([[1.0], [0.0]], dtype=complex)
KET1 = np.array([[0.0], [1.0]], dtype=complex)
KET_PLUS = (KET0 + KET1) / np.sqrt(2)
KET_MINUS = (KET0 - KET1) / np.sqrt(2)
PROJECTORS = {
    Z_BASIS: [KET0 @ KET0.T.conj(), KET1 @ KET1.T.conj()],
    X_BASIS: [KET_PLUS @ KET_PLUS.T.conj(), KET_MINUS @ KET_MINUS.T.conj()],
}
STATE_MAP = {
    (0, Z_BASIS): PROJECTORS[Z_BASIS][0],
    (1, Z_BASIS): PROJECTORS[Z_BASIS][1],
    (0, X_BASIS): PROJECTORS[X_BASIS][0],
    (1, X_BASIS): PROJECTORS[X_BASIS][1],
}


def measure_density(rho: np.ndarray, basis: str) -> Tuple[int, np.ndarray]:
    probs = [float(np.real(np.trace(P @ rho))) for P in PROJECTORS[basis]]
    probs = np.maximum(probs, 0)
    probs = probs / np.sum(probs)
    outcome = int(np.random.choice([0, 1], p=probs))
    post = PROJECTORS[basis][outcome]
    return outcome, post


def fast_bb84_qber(n_qubits: int = 5000, eve: bool = False, noise: float = 0.0) -> float:
    alice_bits = random_bits(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)
    mismatches = 0
    sifted = 0
    mixed = np.eye(2, dtype=complex) / 2
    for bit, a_basis, b_basis in zip(alice_bits, alice_bases, bob_bases):
        rho = STATE_MAP[(bit, a_basis)].copy()
        if eve:
            eve_basis = random.choice(BASES)
            eve_bit, rho = measure_density(rho, eve_basis)
            rho = STATE_MAP[(eve_bit, eve_basis)].copy()
        if random.random() < noise:
            rho = mixed.copy()
        bob_bit, _ = measure_density(rho, b_basis)
        if a_basis == b_basis:
            sifted += 1
            mismatches += int(bit != bob_bit)
    return mismatches / sifted if sifted else 0.0

print("Fast clean QBER:", round(100 * fast_bb84_qber(eve=False), 2), "%")
print("Fast Eve QBER:", round(100 * fast_bb84_qber(eve=True), 2), "%")

## 8. Optional IBM Quantum Runtime Path

The safest hackathon approach is to use the local Aer simulator for the full BB84 demo and reserve real hardware for a **small proof-of-concept**. Real devices introduce noise, queue delays, and measurement error; they are excellent for showing that the project is IBM Quantum-ready, but the educational protocol logic should be debugged locally first.

The cell below is intentionally optional. It uses `getpass` so you do not paste tokens into the notebook or repository.

In [ ]:
# Optional IBM Quantum setup. Leave this cell commented until you need real hardware.

# from getpass import getpass
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler import generate_preset_pass_manager
#
# token = getpass("Enter IBM Quantum API token: ")
# QiskitRuntimeService.save_account(channel="ibm_quantum", token=token, overwrite=True)
# service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)
# print("Selected backend:", backend.name)
#
# # Small single-qubit proof-of-concept: Alice encodes |+>, Bob measures in X.
# qc = encode_bit(0, X_BASIS)
# qc = add_measurement_in_basis(qc, X_BASIS)
# pass_manager = generate_preset_pass_manager(optimization_level=1, backend=backend)
# isa_circuit = pass_manager.run(qc)
# sampler = Sampler(mode=backend)
# job = sampler.run([isa_circuit], shots=256)
# print("Job ID:", job.job_id())
# result = job.result()
# print(result[0].data.c.get_counts() if hasattr(result[0].data, "c") else result[0].data)

## 9. Judge-Ready Takeaways

This notebook provides a complete quantum foundation for the BB84 educational interface. The most important demo moment is the contrast between the clean run and Eve run.

| Demo Claim | Evidence Generated by Notebook |
|---|---|
| Alice and Bob can establish a shared key when no one interferes | Clean run has low QBER and matching final keys |
| Eve is detected through quantum disturbance | Intercept-resend raises QBER toward the 25 percent region |
| QBER is an intuitive security metric | The sweep plot shows acceptance dropping as noise or Eve increases |
| The project is IBM-ready | Optional Qiskit Runtime section provides a secure real-hardware path |
| The code can support a frontend | Structured result objects, summary dictionaries, and pandas tables are Streamlit-friendly |

### Suggested IBM Bob follow-up prompt

```text
Use this notebook as the validated quantum backend for my BB84 Streamlit educational interface. Convert the run_bb84 result summary, protocol trace table, and Monte Carlo QBER sweep into frontend components. Keep Aer simulation as the default and expose IBM Quantum Runtime as an optional advanced panel with secure token entry.
```

In [ ]:
# Export a compact JSON-like summary for frontend integration.
summary_payload = {
    "clean": clean_run.summary(),
    "eve": eve_run.summary(),
    "threshold_percent": SECURITY_THRESHOLD * 100,
    "teaching_message": "If QBER rises above the threshold, Alice and Bob reject the key because the quantum channel may be compromised.",
}
summary_payload

## 10. Security Interpretation: What the Notebook Is Proving

This notebook is an educational implementation, not a production cryptosystem. Its strongest value is that it turns the abstract BB84 security story into measurable artifacts: basis agreement, sifted-key length, sample disclosure, QBER, and key acceptance. In a real QKD system, Alice and Bob would also authenticate the public channel, perform more rigorous parameter estimation, run interactive error correction, and apply privacy amplification before using the key.

| Security Component | What This Notebook Implements | What a Production QKD System Adds |
|---|---|---|
| Quantum encoding | Z and X basis BB84 states | Calibrated optical pulses or device-specific quantum states |
| Eavesdropper visibility | Intercept-resend raises QBER | Formal finite-key security proof and device assumptions |
| Public discussion | Basis sifting and sampled-bit comparison | Authenticated classical channel |
| Error correction | Simplified demonstration section below | Information reconciliation such as Cascade or LDPC |
| Privacy amplification | SHA-256 teaching compression below | Universal hashing with quantified leakage bounds |

The judging takeaway is that **QBER is the bridge between quantum behavior and cybersecurity decision-making**. The frontend can show that Alice and Bob do not need to identify Eve directly; they only need to observe that the channel statistics are no longer compatible with a safe key exchange.

In [ ]:
def qber_decision_report(result: BB84Result, threshold: float = SECURITY_THRESHOLD) -> str:
    """Create a judge-friendly paragraph from a BB84 result."""
    status = "ACCEPT" if result.qber <= threshold else "REJECT"
    reason = "within" if result.qber <= threshold else "above"
    return (
        f"Decision: {status} key. The measured QBER is {100*result.qber:.2f}%, "
        f"which is {reason} the {100*threshold:.1f}% teaching threshold. "
        f"Alice and Bob kept {result.final_key_length} final bits after sifting and sampling. "
        f"Eve mode: {result.eve_strategy}. Channel noise: {result.channel_noise:.3f}."
    )

print(qber_decision_report(clean_run))
print(qber_decision_report(eve_run))

## 11. Simplified Error Correction and Privacy Amplification

After QBER estimation, a real QKD protocol does not immediately use the raw sifted key. Alice and Bob first reconcile errors and then compress the key so any information leaked during reconciliation becomes useless to Eve. The following functions are intentionally compact and pedagogical. They are useful for a hackathon interface because they show the **post-processing pipeline** after the quantum stage.

In [ ]:
import hashlib


def parity(bits: List[int]) -> int:
    return sum(bits) % 2


def simple_block_reconcile(alice_key: List[int], bob_key: List[int], block_size: int = 8) -> Tuple[List[int], Dict[str, int]]:
    """Educational one-pass block parity reconciliation.

    If a block parity differs, this demo flips Bob's first mismatched bit in that block.
    A production Cascade implementation would use multiple passes and binary search.
    """
    corrected = bob_key.copy()
    leaked_parity_bits = 0
    corrected_errors = 0
    for start in range(0, min(len(alice_key), len(bob_key)), block_size):
        end = min(start + block_size, len(alice_key), len(bob_key))
        a_block = alice_key[start:end]
        b_block = corrected[start:end]
        leaked_parity_bits += 1
        if parity(a_block) != parity(b_block):
            for offset, (a, b) in enumerate(zip(a_block, b_block)):
                if a != b:
                    corrected[start + offset] = a
                    corrected_errors += 1
                    break
    return corrected, {"leaked_parity_bits": leaked_parity_bits, "corrected_errors": corrected_errors}


def privacy_amplify(bits: List[int], output_bits: int = 128) -> str:
    """Compress a reconciled bit list into a hex key for demonstration."""
    bit_string = ''.join(str(b) for b in bits)
    digest = hashlib.sha256(bit_string.encode('utf-8')).hexdigest()
    hex_chars = max(1, min(len(digest), math.ceil(output_bits / 4)))
    return digest[:hex_chars]

# Demonstrate post-processing on a clean accepted run.
reconciled_bob, reconcile_stats = simple_block_reconcile(clean_run.alice_final_key, clean_run.bob_final_key)
final_demo_key = privacy_amplify(clean_run.alice_final_key, output_bits=128)
print("Reconciliation stats:", reconcile_stats)
print("Bob corrected matches Alice:", reconciled_bob == clean_run.alice_final_key)
print("Demo privacy-amplified key:", final_demo_key)

## 12. Key-Rate Estimation for the Frontend

A strong educational interface should not only show whether a key is accepted; it should estimate how many usable secret bits remain. The formula below is simplified, but it is excellent for a dashboard because learners can see how basis mismatch, QBER sampling, reconciliation leakage, and privacy amplification reduce the final key.

In [ ]:
def binary_entropy(p: float) -> float:
    if p <= 0 or p >= 1:
        return 0.0
    return -p * math.log2(p) - (1 - p) * math.log2(1 - p)


def estimate_teaching_secret_fraction(qber: float, reconciliation_efficiency: float = 1.2) -> float:
    """Approximate secret fraction: 1 - f*h(QBER) - h(QBER), clipped to [0,1]."""
    h = binary_entropy(qber)
    return max(0.0, 1.0 - reconciliation_efficiency * h - h)


def key_rate_dashboard_row(result: BB84Result) -> Dict[str, float]:
    secret_fraction = estimate_teaching_secret_fraction(result.qber)
    return {
        "eve_strategy": result.eve_strategy,
        "qber_percent": 100 * result.qber,
        "sifted_bits": result.sifted_key_length,
        "sampled_bits": len(result.sample_positions),
        "raw_final_bits": result.final_key_length,
        "estimated_secret_fraction": secret_fraction,
        "estimated_secret_bits": result.final_key_length * secret_fraction,
    }

rate_df = pd.DataFrame([key_rate_dashboard_row(clean_run), key_rate_dashboard_row(eve_run)])
display(rate_df)

fig, ax = plt.subplots(figsize=(8, 4))
q = np.linspace(0.0, 0.25, 100)
ax.plot(100*q, [estimate_teaching_secret_fraction(float(x)) for x in q])
ax.axvline(100*SECURITY_THRESHOLD, color='red', linestyle='--', label='teaching threshold')
ax.set_title('Simplified BB84 secret fraction vs QBER')
ax.set_xlabel('QBER (%)')
ax.set_ylabel('Estimated secret fraction')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 13. Batch Circuit Construction for Better Qiskit Performance

The earlier cells use one circuit per qubit because that is easiest to teach. For a more impressive Qiskit foundation, the function below builds a single multi-qubit circuit where each qubit represents one BB84 transmission. This is efficient for clean and noisy simulation when Eve is not modeled by mid-circuit measurement. It is also a good architecture pattern for future frontend optimization.

In [ ]:
def build_parallel_bb84_circuit(alice_bits: List[int], alice_bases: List[str], bob_bases: List[str]) -> QuantumCircuit:
    """Build one multi-qubit BB84 circuit for all transmissions without Eve."""
    n = len(alice_bits)
    if not (len(alice_bases) == len(bob_bases) == n):
        raise ValueError('bit and basis lists must have equal length')
    qc = QuantumCircuit(n, n, name='parallel_bb84')
    for i, (bit, basis) in enumerate(zip(alice_bits, alice_bases)):
        if bit == 1:
            qc.x(i)
        if basis == X_BASIS:
            qc.h(i)
    qc.barrier()
    for i, basis in enumerate(bob_bases):
        if basis == X_BASIS:
            qc.h(i)
    qc.measure(range(n), range(n))
    return qc


def run_parallel_clean_bb84(n_qubits: int = 16) -> Dict[str, object]:
    alice_bits = random_bits(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)
    qc = build_parallel_bb84_circuit(alice_bits, alice_bases, bob_bases)
    tqc = transpile(qc, SIMULATOR, seed_transpiler=RNG_SEED)
    counts = SIMULATOR.run(tqc, shots=1).result().get_counts()
    bitstring = max(counts, key=counts.get)[::-1]  # reverse to align classical bit i with index i
    bob_bits = [int(bitstring[i]) for i in range(n_qubits)]
    sifted = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    errors = sum(alice_bits[i] != bob_bits[i] for i in sifted)
    return {"circuit": qc, "sifted_indices": sifted, "qber": errors / len(sifted) if sifted else 0.0, "counts": counts}

parallel_demo = run_parallel_clean_bb84(12)
print("Parallel circuit QBER:", parallel_demo['qber'])
parallel_demo['circuit'].draw(output='mpl')

## 14. Aer Noise Model Extension

The simple bit-flip model is easy to teach, but Qiskit Aer can also attach gate and readout errors to a simulator. This section gives you a realistic extension point without making the notebook depend on live IBM hardware. It is intentionally small, correct, and safe to run locally.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, readout_error


def make_educational_noise_model(single_qubit_error: float = 0.002, readout_p01: float = 0.01, readout_p10: float = 0.01) -> NoiseModel:
    """Create a small Aer noise model for one-qubit BB84 circuits."""
    noise_model = NoiseModel()
    one_q_error = depolarizing_error(single_qubit_error, 1)
    noise_model.add_all_qubit_quantum_error(one_q_error, ['x', 'h'])
    ro_error = readout_error.ReadoutError([[1 - readout_p01, readout_p01], [readout_p10, 1 - readout_p10]])
    noise_model.add_all_qubit_readout_error(ro_error)
    return noise_model


def measure_with_noise(circuit: QuantumCircuit, basis: str, noise_model: NoiseModel) -> int:
    noisy_sim = AerSimulator(noise_model=noise_model, seed_simulator=RNG_SEED)
    measured = add_measurement_in_basis(circuit, basis)
    tqc = transpile(measured, noisy_sim, seed_transpiler=RNG_SEED)
    counts = noisy_sim.run(tqc, shots=1).result().get_counts()
    return int(max(counts, key=counts.get))

noise_model = make_educational_noise_model()
example_counts_circuit = add_measurement_in_basis(encode_bit(0, X_BASIS), X_BASIS)
noisy_sim = AerSimulator(noise_model=noise_model, seed_simulator=RNG_SEED)
counts = noisy_sim.run(transpile(example_counts_circuit, noisy_sim), shots=512).result().get_counts()
plot_histogram(counts)

## 15. Streamlit and IBM Bob Integration Contract

The notebook is easiest to integrate into the existing educational interface if each major computation returns a plain dictionary. The contract below is a recommended API shape for IBM Bob or a future backend service.

| Field | Type | Frontend Usage |
|---|---|---|
| `summary` | dictionary | metric cards and acceptance banner |
| `trace_rows` | list of dictionaries | protocol explanation table |
| `decision_report` | string | judge-friendly explanation |
| `rate_estimate` | dictionary | key-rate and QBER dashboard |
| `final_key_preview` | string | safe truncated demo key display |

In [ ]:
def notebook_to_frontend_payload(result: BB84Result, preview_bits: int = 32) -> Dict[str, object]:
    trace = show_protocol_trace(result, rows=20).to_dict(orient='records')
    return {
        "summary": result.summary(),
        "trace_rows": trace,
        "decision_report": qber_decision_report(result),
        "rate_estimate": key_rate_dashboard_row(result),
        "final_key_preview": ''.join(map(str, result.alice_final_key[:preview_bits])),
        "warning": "Preview is truncated for demo safety; production systems must never expose secret keys in UI logs.",
    }

frontend_payload = notebook_to_frontend_payload(eve_run)
frontend_payload

## 16. Demo Script for the Hackathon Presentation

Start by running the clean BB84 simulation and point out that basis mismatch is normal, but matching-basis bits agree. Next, enable Eve and rerun. The most important visual is the QBER chart: the clean channel stays low while intercept-resend jumps upward. Then show the decision report and explain that Alice and Bob reject the key before using it. Finally, open the optional IBM Quantum section and explain that the same one-qubit gates, `X`, `H`, and measurement, are compatible with real IBM Quantum hardware for a small proof-of-concept.

> **One-sentence pitch:** This project turns quantum cryptography from a textbook protocol into an interactive security lab where learners can see eavesdropping become measurable noise.

## 17. Extended Educational Narrative: Why BB84 Matters

BB84 is one of the clearest ways to teach the difference between classical and quantum information security. A classical cryptographic system normally depends on computational assumptions, such as the difficulty of factoring large numbers or solving discrete logarithm problems. BB84 is different in spirit because the security lesson is tied to physical behavior: a measurement in the wrong basis changes the state being measured. This notebook uses that physical behavior as the central learning object. When Eve attempts to learn the key by intercepting qubits, she cannot simply copy them and pass perfect duplicates to Bob. Her measurement forces a basis choice, and the wrong choice creates detectable errors.

For an educational interface, this distinction is extremely powerful. Learners do not need to begin with advanced mathematics or a full security proof. They can begin with a simple question: **what happens when a third party tries to observe the quantum channel?** The notebook answers this question through the sequence of Alice preparation, Bob measurement, public basis comparison, QBER estimation, and acceptance or rejection. Each step is represented in code and also produces concrete data that a frontend can display.

A hackathon judge should be able to understand the value of the project in under one minute. The notebook supports that by turning BB84 into a measurable story. In a clean simulation, Alice and Bob keep the bits where they chose the same basis, and those bits match. In an attacked simulation, Eve's intervention increases QBER. The security decision is not based on guessing Eve's identity or intent; it is based on measured disturbance. This is the core intuition behind the project.

| Learning Question | Notebook Mechanism | Visible Output |
|---|---|---|
| How does Alice encode information? | `encode_bit` prepares Z-basis and X-basis states | Circuit diagrams and protocol trace table |
| Why does basis choice matter? | Bob randomly measures in Z or X | Matching-basis rows become sifted key rows |
| How is Eve detected? | `intercept_resend` measures and resends | QBER increases in Eve simulations |
| How is a security decision made? | `qber_decision_report` compares QBER to threshold | Accept or reject explanation |
| How does this become a product? | Frontend payload function returns dictionaries | Streamlit-ready metrics and tables |

## 18. Detailed Protocol Walkthrough for Non-Quantum Audiences

Imagine Alice wants to share a secret string of bits with Bob. If she sends the bits as ordinary classical messages, an eavesdropper can copy them without leaving evidence. BB84 changes the communication medium. Alice does not send only a classical bit; she sends a quantum state chosen from two possible bases. The bit value and the basis together define the state. Bob does not know the basis ahead of time, so he randomly chooses how to measure. Sometimes Bob chooses the same basis as Alice, and sometimes he does not.

After transmission, Alice and Bob publicly compare only their basis choices. They do not reveal the secret bit values except for a small sample used to estimate QBER. Whenever their bases match, Bob's measurement should agree with Alice's bit in an ideal channel. Whenever their bases differ, Bob's result is not reliable, so those positions are discarded. This public comparison process is called **sifting**.

The important security insight appears when Eve enters the channel. Eve also does not know Alice's basis. If Eve guesses correctly, she can resend a compatible state. If she guesses incorrectly, she collapses the state in the wrong basis and resends a state that may disagree with Alice's original bit when Bob later measures in the correct basis. Because Eve cannot know in advance which basis Alice used, her measurements create a statistical footprint.

The notebook exposes this footprint through QBER. QBER is the fraction of sampled sifted bits where Alice and Bob disagree. If the observed QBER is small, the notebook accepts the key in a teaching sense. If QBER is too high, it rejects the key. This is exactly the type of concept that works well in a classroom or hackathon demo because it connects quantum mechanics, cybersecurity, and data visualization in one workflow.

## 19. Mathematical Intuition Without Heavy Formalism

The intercept-resend attack has a memorable expected error rate. Eve chooses the correct basis about half of the time. When she chooses the correct basis, she learns the bit and can resend the correct state, so she does not introduce an error in those positions. When she chooses the wrong basis, her measurement result is random relative to Alice's original state. If Bob later uses Alice's correct basis, the state Eve resent gives Bob the wrong bit about half of the time.

The sifted key only includes positions where Bob's basis matched Alice's basis. Therefore, among the sifted positions, Eve is wrong about the basis roughly half of the time, and those wrong-basis interventions create an error roughly half of the time. Multiplying one half by one half gives an expected QBER around one quarter, or 25 percent. Randomness means one run may not be exactly 25 percent, but the Monte Carlo section makes the average pattern visible.

| Event | Approximate Probability | Effect |
|---|---:|---|
| Bob uses Alice's basis | 1/2 | Bit can enter sifted key |
| Eve uses Alice's basis | 1/2 | No disturbance from Eve's measurement |
| Eve uses wrong basis | 1/2 | State is disturbed |
| Bob gets wrong bit after Eve's wrong basis | 1/2 | Sifted-key error |
| Expected QBER on sifted bits | 1/4 | Eve becomes detectable |

This explanation is intentionally accessible. It is not a finite-key proof, and it does not cover implementation loopholes, detector attacks, photon-number splitting, or authentication requirements. However, it is the right level for an educational interface because learners first need to understand why measurement itself can become a security signal.

## 20. Notebook-to-Product Design Rationale

The code in this notebook is structured so it can serve as the backend logic for a larger product. A hackathon prototype should be easy to explain, easy to extend, and easy to demonstrate. For that reason, the notebook separates the simulation into readable functions rather than hiding everything inside one long cell. The functions `encode_bit`, `measure_one_shot`, `intercept_resend`, `run_bb84`, `monte_carlo_sweep`, and `notebook_to_frontend_payload` each correspond to a feature that can be surfaced in a user interface.

A strong frontend would not simply show raw code outputs. It would guide the learner through a narrative. First, the user selects the number of qubits and whether Eve is active. Second, the app displays a visual representation of Alice's random choices. Third, it reveals Bob's basis choices and shows which positions survive sifting. Fourth, it samples part of the sifted key and computes QBER. Finally, it announces whether the key is accepted or rejected. This notebook already produces every data structure needed for that flow.

| Product Component | Backing Notebook Function | Suggested Interface Element |
|---|---|---|
| Alice state preparation | `encode_bit` | Circuit card for each basis-bit pair |
| Protocol simulation | `run_bb84` | Run button with sliders for qubits, noise, Eve |
| Eve attack explanation | `intercept_resend` | Toggle with warning panel |
| QBER analytics | `monte_carlo_sweep` | Line chart comparing clean and attacked channels |
| Security decision | `qber_decision_report` | Accept or reject banner |
| Frontend API payload | `notebook_to_frontend_payload` | JSON response for Streamlit or FastAPI |

This architecture also fits the IBM Bob Hackathon workflow. Bob can help refactor notebook cells into modules, create UI components, add tests, or convert the payload into an API endpoint. The notebook remains the authoritative explanation of the quantum logic, while the application becomes the interactive experience.

## 21. Detailed Frontend Storyboard

The educational interface should begin with a short, confident introduction: Alice wants to share a secret key with Bob, and the channel may be watched by Eve. The user should immediately see three controls: the number of qubits, the Eve strategy, and the channel noise level. These controls make the demo interactive without overwhelming the learner. The default run should be clean and successful, because the learner first needs to see what normal behavior looks like.

After the first run, the interface should show a protocol trace table. Each row can contain Alice's bit, Alice's basis, Bob's basis, Bob's bit, whether the bases matched, and whether the row was used for QBER sampling. The table should be visually styled so matched-basis rows are highlighted. This makes sifting intuitive: learners can literally see rows being kept or discarded.

The next panel should show QBER. A gauge or metric card works well here. If QBER is below the threshold, the card can show an accepted state. If QBER is above the threshold, the card can show a rejected state. The interface should emphasize that rejection is a success condition for security: the protocol detected that the channel was not safe.

The final panel should show the Monte Carlo comparison. This is the judge-facing proof that the project is not just one lucky random run. The chart should compare clean, noisy, and attacked cases. It should show that Eve produces a consistent rise in QBER, and that excessive ordinary noise can also force rejection. That dual lesson is important because QKD systems must distinguish ideal theory from realistic channels.

## 22. Suggested Streamlit Layout in Detail

A practical Streamlit implementation can use four tabs: **Protocol Lab**, **Eve Attack Lab**, **QBER Analytics**, and **IBM Quantum Mode**. The Protocol Lab should be the default tab. It should contain the basic controls and a simple run button. The Eve Attack Lab should explain intercept-resend with diagrams and before-after examples. The QBER Analytics tab should contain the Monte Carlo sweep and key-rate estimate. The IBM Quantum Mode tab should remain optional and should clearly warn users not to paste secrets into code cells or commit tokens.

| Tab | Primary User Goal | Notebook Source |
|---|---|---|
| Protocol Lab | Understand Alice, Bob, bases, and sifting | `run_bb84`, `show_protocol_trace` |
| Eve Attack Lab | Observe how Eve changes the statistics | `intercept_resend`, clean vs Eve summaries |
| QBER Analytics | Compare scenarios across many trials | `monte_carlo_sweep`, secret fraction plot |
| IBM Quantum Mode | Connect concept to IBM ecosystem | optional Runtime section |
| Build Notes | Help future contributors extend the project | integration contract and demo script |

The frontend should avoid exposing the full final key by default. It can show a short preview and clearly label it as a demonstration artifact. In a real cryptographic system, secret keys should never be printed to logs or displayed in a public interface. This notebook already includes a warning field in the frontend payload for that reason.

## 23. IBM Quantum Hardware Discussion for Judges

The notebook deliberately uses Aer as the default backend because it produces fast and repeatable results. That decision is important for hackathon reliability. Live hardware is valuable, but it introduces queue times, calibration drift, readout errors, gate noise, and shot limitations. A polished demo should not depend on a queue clearing at the right time. Instead, the project should use Aer for the complete educational flow and use IBM Quantum hardware for a small optional proof that the same circuit primitives are compatible with real devices.

The optional hardware section focuses on a single-qubit example rather than attempting to run the full BB84 flow on a live backend. This is a reasonable design choice because BB84 communication is fundamentally a prepare-and-measure protocol, while cloud quantum hardware is optimized for circuit execution. The notebook still demonstrates the relevant operations: preparing `|0>`, `|1>`, `|+>`, and `|->`, applying `H` for basis changes, and measuring.

For a judging presentation, the best wording is: **the educational protocol is simulator-first for reliability, but the quantum primitives are Qiskit-native and IBM Quantum-ready**. That phrase honestly communicates both the strength and the boundary of the implementation.

## 24. Error Correction Explanation for Learners

Once Alice and Bob estimate QBER and decide the channel is acceptable, they still may have small differences in their sifted keys. Error correction is the process of reconciling those differences over the public channel. The public discussion leaks some information, so it must be accounted for later. The simplified block parity function in this notebook is not a production reconciliation protocol, but it illustrates the basic idea: Alice and Bob compare parity values for blocks of bits, and parity mismatches indicate that at least one error exists in the block.

A classroom analogy is useful. Suppose Alice and Bob each have a row of eight secret bits. They do not reveal the bits, but they reveal whether the number of ones in the row is even or odd. If both parities match, the row may be correct, though two errors could still hide. If parities differ, the row definitely contains an odd number of errors. A real protocol uses more sophisticated searching and multiple passes. The notebook's simplified function flips the first known mismatch only because this is a teaching environment where both arrays are available to the simulator.

The important message for learners is that error correction is not free. Every public parity bit leaks a small amount of information. That leakage motivates privacy amplification, where the reconciled key is compressed to reduce Eve's possible knowledge.

## 25. Privacy Amplification Explanation for Learners

Privacy amplification is the final transformation from a partially secret reconciled key into a shorter, stronger secret key. The notebook uses SHA-256 as a teaching compression function because it is familiar and easy to demonstrate. A production QKD system would use privacy amplification based on universal hashing and would carefully compute the output length from the security proof, observed QBER, reconciliation leakage, and finite-size effects.

The educational value is still high. Learners can see that the final key is shorter than the raw sifted key. This is a critical lesson: QKD does not magically turn every transmitted qubit into a usable secret bit. Some bits are discarded due to basis mismatch. Some are revealed for QBER testing. Some effective secrecy is lost during reconciliation. Privacy amplification compresses what remains.

| Stage | Why Bits Are Lost | Educational Meaning |
|---|---|---|
| Transmission | Bob chooses wrong basis for some qubits | Random bases protect against undetected observation |
| Sifting | Mismatched bases are discarded | Public discussion keeps only reliable positions |
| Sampling | Some sifted bits are revealed | QBER must be estimated before trusting the key |
| Reconciliation | Parity and correction data leak information | Correctness has a security cost |
| Privacy amplification | Key is compressed | Eve's partial knowledge is reduced |

This staged reduction is one of the best visuals for the frontend. A funnel chart would be effective: transmitted qubits enter at the top, final secret bits leave at the bottom.

## 26. Testing and Validation Strategy

A hackathon notebook becomes much stronger when it includes a clear validation story. The core correctness checks are simple. In a clean ideal simulation, QBER should usually be near zero. With intercept-resend, average QBER should be much higher. The final key lengths should be less than the sifted key lengths because some bits are sampled. The accepted flag should generally be true for clean low-noise runs and false for strong Eve runs.

A future repository should convert these expectations into automated tests. For example, a test can run twenty clean simulations and assert that average QBER remains low. Another test can run twenty Eve simulations and assert that average QBER exceeds a threshold such as 15 percent. Because quantum measurement simulation is random, tests should use ranges and averages rather than exact values. The notebook already fixes random seeds to make examples stable, but robust testing should still account for probabilistic variation.

| Test Name | Expected Behavior | Why It Matters |
|---|---|---|
| clean channel QBER | Average QBER near zero | Confirms encoding and basis measurement are aligned |
| Eve QBER increase | Eve average QBER much higher than clean | Confirms attack model creates disturbance |
| sifting length | Around half of transmissions survive | Confirms random basis comparison works |
| sampling removal | Final key shorter than sifted key | Confirms QBER sample is not reused as secret |
| frontend payload shape | Required keys are present | Confirms app integration stability |

## 27. Common Demo Failure Modes and How to Explain Them

Randomized simulations sometimes produce surprising individual runs. A clean run may show a small nonzero QBER if noise is enabled. An Eve run may occasionally show less than 25 percent QBER because the sample is finite. This is not a bug; it is a teaching opportunity. The correct explanation is that QBER is a statistical estimate. Larger numbers of qubits and repeated trials produce more stable averages.

Another common issue is confusion between transmitted bits, sifted bits, sampled bits, and final key bits. The interface should avoid using the phrase "the key" too early. The transmitted random bits are not yet the final key. The sifted key is a candidate key. The sampled bits are sacrificed to estimate security. The final key is what remains after sampling and post-processing.

If the IBM Quantum hardware cell is used, queue delays or noisy results may occur. The demo should frame hardware execution as optional. The primary educational proof should rely on the local simulator, while the hardware section demonstrates ecosystem compatibility.

## 28. Ethical and Security Notes

This notebook should not be presented as production-ready cryptographic software. It is an educational prototype. Real QKD deployments require authenticated classical communication, hardware security assumptions, side-channel analysis, device calibration, finite-key analysis, and carefully implemented post-processing. The notebook intentionally simplifies these topics so that the core concept is understandable.

The interface should also avoid giving learners the impression that quantum cryptography replaces all cybersecurity practices. QKD addresses a specific key-distribution problem under specific assumptions. It does not secure endpoints, prevent malware, authenticate users by itself, or protect data after a key is misused. A strong hackathon presentation should say this clearly because it builds trust with technical judges.

The safest claim is: **this project is a rigorous educational simulator and Qiskit foundation for BB84, not a deployed security product**. That wording is accurate, professional, and defensible.

## 29. Advanced Extension Ideas for IBM Bob

IBM Bob can be used to extend this notebook into a full project. The highest-value extension is to convert the simulation functions into a Python package with tests. The next extension is a Streamlit or React frontend that calls the simulation and displays the results. A third extension is an optional FastAPI service that returns frontend payloads. A fourth extension is a notebook export mode that produces a Markdown or PDF lab report for students.

| Extension | Difficulty | Impact | Suggested Bob Prompt |
|---|---:|---:|---|
| Package refactor | Medium | High | Refactor notebook functions into `cryptolab/bb84_qiskit.py` with tests |
| Streamlit dashboard | Medium | Very high | Build tabs for protocol trace, QBER analytics, and Eve attack visualization |
| FastAPI backend | Medium | High | Expose `/simulate`, `/sweep`, and `/payload` endpoints |
| Student worksheet | Low | Medium | Generate a fill-in-the-blank lab worksheet from the notebook narrative |
| Hardware proof cell | Medium | Medium | Add backend selection and safe token handling for IBM Quantum Runtime |
| Visualization polish | Low | High | Add funnel chart, QBER gauge, and basis-match heatmap |

The most practical hackathon path is to keep the quantum logic in Python, use Streamlit for fast UI iteration, and document IBM Quantum Runtime as an optional advanced feature. This balances ambition with reliability.

## 30. Expanded Judge Pitch

This project is an educational quantum cybersecurity lab built around BB84 Quantum Key Distribution. It uses Qiskit to show how Alice encodes random bits into quantum states, how Bob measures in random bases, and how public basis comparison creates a sifted key. The project then introduces Eve through an intercept-resend attack and demonstrates that her attempt to observe the channel creates measurable errors. The central metric is QBER, which becomes the learner's bridge between quantum measurement and practical security decisions.

The notebook is valuable because it is not only explanatory; it is executable. Every major claim is backed by code. The clean simulation shows low QBER. The Eve simulation raises QBER. The Monte Carlo sweep demonstrates that the effect is statistical and repeatable. The post-processing sections explain why real QKD systems need reconciliation and privacy amplification after the quantum stage. The optional IBM Quantum section shows that the project is aligned with IBM's quantum software ecosystem while keeping the main demo reliable on a local simulator.

For the hackathon, the most compelling live flow is to run the clean case, ask the audience whether the key should be trusted, then turn on Eve and run the same protocol. The interface should show the QBER jump and reject the key. That moment makes the idea memorable: Eve does not need to be caught directly; the quantum channel reveals that something changed.

## 31. Detailed Glossary for the Notebook

| Term | Explanation |
|---|---|
| Alice | The sender who prepares random bits in random quantum bases |
| Bob | The receiver who measures each qubit in a random basis |
| Eve | The eavesdropper who tries to learn the key by intercepting the quantum channel |
| Basis | A measurement framework; this notebook uses Z and X bases |
| Z basis | The computational basis with states `|0>` and `|1>` |
| X basis | The Hadamard basis with states `|+>` and `|->` |
| Sifting | Publicly comparing bases and keeping only matching-basis positions |
| QBER | Quantum Bit Error Rate; the fraction of sampled sifted bits that disagree |
| Intercept-resend | An eavesdropping strategy where Eve measures and sends a replacement qubit |
| Reconciliation | Error correction process applied after QBER estimation |
| Privacy amplification | Compressing a reconciled key to reduce Eve's possible information |
| Aer | Qiskit's high-performance simulator package used for local quantum circuit simulation |
| Runtime | IBM Quantum service pathway for executing circuits on IBM backends |

This glossary can be copied directly into the frontend as a help panel. It reduces friction for non-quantum judges and makes the demo more accessible to students.

## 32. Long-Form Classroom Script

Begin the lesson by saying that Alice and Bob are not trying to send a secret message directly. They are trying to create a shared secret key. Once they have that key, they could use it with a classical encryption method. The quantum part is the key distribution stage. This distinction matters because it prevents learners from thinking that the qubits themselves are the final encrypted message.

Next, introduce the two bases. Explain that Alice chooses both a bit and a basis randomly. If she chooses bit 0 in the Z basis, she sends `|0>`. If she chooses bit 1 in the Z basis, she sends `|1>`. If she chooses bit 0 in the X basis, she sends `|+>`. If she chooses bit 1 in the X basis, she sends `|->`. At this point, show the circuit examples in the notebook. The X gate changes `|0>` to `|1>`, and the H gate changes the measurement basis relationship.

Then let Bob measure. Bob does not know Alice's basis, so he guesses randomly. When Bob guesses correctly, he gets the right bit in an ideal channel. When Bob guesses incorrectly, the result is random and is not useful for the final key. Alice and Bob publicly compare bases and discard mismatches. Stress that they are not revealing all of their bit values.

Now introduce Eve. Eve has the same problem as Bob: she does not know Alice's basis. But Eve's action is more dangerous because she measures before Bob receives the qubit. If she chooses wrong, she disturbs the state. When Bob later measures in Alice's original basis, the disturbance can become an error. Alice and Bob reveal a sample of their sifted bits, compute QBER, and reject the key if the error rate is too high.

End by connecting this to the larger cybersecurity story. Classical eavesdropping can be passive and invisible. BB84 makes observation physically consequential. That is the intuition the project is designed to teach.

## 33. Future Research and Product Roadmap

The current notebook is already strong enough to serve as the quantum foundation for a hackathon prototype, but it can support a much larger roadmap. The first research extension is finite-key analysis. Real systems cannot assume infinite data, so they must bound Eve's information from a finite sample. The second extension is realistic optical-channel modeling, including loss, detector efficiency, and dark counts. The third extension is authentication of the classical channel, because BB84 assumes Alice and Bob can trust that their public discussion is not being modified by an active man-in-the-middle.

On the product side, the most valuable roadmap item is an interactive learning path. A beginner mode can hide equations and focus on visual intuition. An intermediate mode can show QBER and Monte Carlo sweeps. An advanced mode can expose reconciliation, privacy amplification, and IBM Runtime. This layered design helps the same project serve students, judges, and technical reviewers.

| Roadmap Layer | Feature | Value |
|---|---|---|
| Beginner | Animated Alice-Bob-Eve flow | Makes the protocol memorable |
| Intermediate | QBER sweep dashboard | Shows statistical detection |
| Advanced | Reconciliation and privacy amplification | Connects toy protocol to real QKD pipeline |
| Expert | Finite-key and hardware noise models | Supports deeper technical credibility |
| Deployment | Streamlit, FastAPI, and Docker | Makes the project easy to run and judge |

## 34. Quantum Mechanics Deep Dive: State Vectors, Bases, and the Born Rule

BB84 is powerful because it uses the fact that quantum information depends on the measurement basis. A single qubit can be represented as a vector in a two-dimensional complex Hilbert space. The computational basis contains `|0>` and `|1>`, while the Hadamard basis contains `|+> = (|0> + |1>)/sqrt(2)` and `|-> = (|0> - |1>)/sqrt(2)`. These two bases are not just different labels; they represent incompatible measurement contexts.

The **Born rule** says that the probability of a measurement outcome is the squared magnitude of the amplitude projected onto the measurement basis. In BB84, this means measuring `|0>` in the Z basis gives 0 with probability 1, but measuring `|0>` in the X basis gives `+` or `-` with equal probability. This is exactly why Bob discards mismatched-basis measurements and why Eve cannot safely measure without knowing Alice's basis.

| Prepared State | Z Measurement | X Measurement | BB84 Meaning |
|---|---|---|---|
| `|0>` | 0 with probability 1 | random | Z-basis bit 0 |
| `|1>` | 1 with probability 1 | random | Z-basis bit 1 |
| `|+>` | random | 0 with probability 1 | X-basis bit 0 |
| `|->` | random | 1 with probability 1 | X-basis bit 1 |

In [ ]:
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit.visualization import plot_bloch_vector

# Show BB84 states as statevectors.
states = {
    "|0>": Statevector.from_label('0'),
    "|1>": Statevector.from_label('1'),
    "|+>": Statevector.from_instruction(encode_bit(0, X_BASIS).remove_final_measurements(inplace=False)),
    "|->": Statevector.from_instruction(encode_bit(1, X_BASIS).remove_final_measurements(inplace=False)),
}
for name, sv in states.items():
    print(name, sv.data)

# Fidelity shows which BB84 states are identical, orthogonal, or partially overlapping.
fidelity_table = pd.DataFrame(
    [[round(state_fidelity(a, b), 3) for b in states.values()] for a in states.values()],
    index=states.keys(), columns=states.keys()
)
display(fidelity_table)

## 35. Bloch Sphere Interpretation of BB84

The Bloch sphere gives a geometric picture of a qubit. The Z-basis states live at the north and south poles, while the X-basis states live on the positive and negative X-axis. BB84 deliberately chooses states from different axes so that a measurement aligned with one axis gives uncertain information about the other axis. This geometric view is one of the best ways to explain why Eve's wrong-basis measurement causes disturbance.

In a frontend, the Bloch sphere can be used as an advanced visualization panel. When Alice selects Z basis, the point appears on the vertical axis. When Alice selects X basis, the point appears on the horizontal axis. Eve's measurement can be animated as a forced projection onto her chosen axis, followed by Bob's final measurement.

In [ ]:
# Bloch vectors for the four BB84 states: |0>, |1>, |+>, |->.
bloch_vectors = {
    "|0> (Z bit 0)": [0, 0, 1],
    "|1> (Z bit 1)": [0, 0, -1],
    "|+> (X bit 0)": [1, 0, 0],
    "|-> (X bit 1)": [-1, 0, 0],
}
for label, vec in bloch_vectors.items():
    print(label, "Bloch vector:", vec)
# In an interactive notebook, run: plot_bloch_vector([1,0,0], title='|+> state')

## 36. No-Cloning Theorem and Why Eve Cannot Copy the Qubits

A common beginner question is why Eve cannot simply copy each qubit, send the original to Bob, and measure her copy later after the bases are announced. The answer is the **no-cloning theorem**: there is no physical quantum operation that can perfectly copy an arbitrary unknown quantum state. BB84 depends on this principle. If Eve could clone unknown states perfectly, she could avoid causing disturbance.

The intuition is linearity. A device that copied `|0>` to `|00>` and copied `|1>` to `|11>` would not also correctly copy a superposition such as `|+>`. Linearity would map `(|0>+|1>)/sqrt(2)` to `(|00>+|11>)/sqrt(2)`, but a true clone would be `|+>|+> = (|00>+|01>+|10>+|11>)/2`. These are different states. Therefore, Eve must choose a measurement strategy, and measurement creates the possibility of detection.

In [ ]:
# A tiny numerical illustration of the no-cloning intuition.
ket00 = np.array([1,0,0,0], dtype=complex)
ket01 = np.array([0,1,0,0], dtype=complex)
ket10 = np.array([0,0,1,0], dtype=complex)
ket11 = np.array([0,0,0,1], dtype=complex)
linear_copy_like = (ket00 + ket11) / np.sqrt(2)       # What linearity would produce from |+>
true_plus_clone = (ket00 + ket01 + ket10 + ket11) / 2 # What perfect cloning of |+> would require
overlap = abs(np.vdot(linear_copy_like, true_plus_clone)) ** 2
print("Fidelity between linearity result and true |+>|+> clone:", round(overlap, 3))
print("Because the fidelity is not 1, a universal perfect cloner cannot exist.")

## 37. Entanglement-Based Perspective: BB84 and E91

BB84 is a prepare-and-measure protocol, but it is closely related to entanglement-based quantum key distribution. In an entanglement-based view, a source distributes entangled pairs to Alice and Bob. When both measure in compatible bases, their outcomes are correlated. Eavesdropping or noise reduces the expected correlations. This perspective is useful because it connects BB84 to Bell tests, entanglement verification, and protocols such as E91.

For this hackathon notebook, the prepare-and-measure version is simpler and more directly useful. However, adding an entanglement demonstration helps judges see that the project is connected to broader quantum information ideas. It also gives a natural path for future expansion: compare BB84 with entanglement-based QKD in a second lab module.

In [ ]:
# Bell-pair demonstration for future entanglement-based QKD extension.
bell = QuantumCircuit(2, 2, name='bell_pair_demo')
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])
counts = SIMULATOR.run(transpile(bell, SIMULATOR), shots=1024).result().get_counts()
print(counts)
plot_histogram(counts)

## 38. Quantum Channel View: Depolarization, Dephasing, and Measurement Error

A more quantum-specific version of the noise story distinguishes among several error types. A bit-flip error applies an `X` operation and swaps `|0>` with `|1>`. A phase-flip error applies `Z`, which leaves computational-basis measurement unchanged but affects superposition states. Depolarizing noise replaces a pure state with a more mixed state. Readout error occurs at measurement time and can make the classical output wrong even if the quantum state was prepared correctly.

This distinction matters for BB84 because the protocol uses more than one basis. A noise process that looks harmless in the Z basis may still damage X-basis states. The use of two conjugate bases therefore makes BB84 sensitive to a broader class of disturbances than a one-basis classical-like protocol.

| Error Type | Quantum Operation | Effect on BB84 |
|---|---|---|
| Bit flip | `X` | Directly changes Z-basis bits and can alter QBER |
| Phase flip | `Z` | Strongly affects X-basis states |
| Depolarizing | random Pauli error | Pushes states toward uncertainty |
| Readout error | classical measurement confusion | Raises observed QBER even without Eve |

In [ ]:
from qiskit_aer.noise import phase_damping_error, phase_amplitude_damping_error

# Example extension point: compare ideal, depolarizing, and phase-related noise models.
def make_phase_noise_model(phase_error_probability: float = 0.01) -> NoiseModel:
    model = NoiseModel()
    model.add_all_qubit_quantum_error(phase_damping_error(phase_error_probability), ['h', 'x'])
    return model

print("Created quantum-specific phase-noise model:", make_phase_noise_model())

## 39. Quantum Measurement as Information Gain Plus Disturbance

The central quantum lesson is that measurement is not passive. In classical computing, reading a bit can be idealized as non-disturbing. In quantum computing, measurement generally changes the state by projecting it into the measurement basis. Eve's intercept-resend attack is therefore not merely a privacy violation; it is a physical intervention in the channel.

This gives the frontend a compelling explanation pattern: **Eve gains information only by paying a disturbance cost**. If Eve measures in the correct basis, the cost may be zero for that qubit. If she measures in the wrong basis, she creates uncertainty that Bob may detect. Across many qubits, the cost becomes visible as QBER.

## References

[1]: https://research.ibm.com/publications/quantum-cryptography-public-key-distribution-and-coin-tossing "Bennett and Brassard, Quantum Cryptography: Public Key Distribution and Coin Tossing"
[2]: https://docs.quantum.ibm.com/ "IBM Quantum Documentation"
[3]: https://qiskit.github.io/qiskit-aer/ "Qiskit Aer Documentation"
[4]: https://docs.quantum.ibm.com/api/qiskit-ibm-runtime "Qiskit IBM Runtime Documentation"